In [ ]:
# ============================================================
# CELL 1: Config
# ============================================================
import os, re
import pandas as pd
from getpass import getuser

USER     = getuser()
DATA_DIR = f"C:/Users/{USER}/Documents/GitHub/tennis-homophily/data/atp"

OLD_FILE = os.path.join(DATA_DIR, "grand_slam_matches_2018_2023.xlsx")
NEW_FILE = os.path.join(DATA_DIR, "grand_slam_matches_2024_2025_olympics.xlsx")
OUT_FILE = os.path.join(DATA_DIR, "grand_slam_matches_2018_2025.xlsx")

# Stage normalisation
# Old data (ATP website):  'Round of 64 ', 'Round of 32 ', 'Round of 16 ',
#                          'Quarter-Finals ', 'Semi-Finals ', 'Finals '
# New data (Wikipedia):    'First Round', 'Second Round', 'Third Round',
#                          'Quarter-Finals', 'Semi-Finals', 'Final'
GS_STAGE_MAP = {          # Australian Open, Roland Garros, Wimbledon (64-team draw)
    'First Round':    'Round of 64',
    'Second Round':   'Round of 32',
    'Third Round':    'Round of 16',
    'Quarter-Finals': 'Quarter-Finals',
    'Semi-Finals':    'Semi-Finals',
    'Final':          'Finals',
}
USO_STAGE_MAP = {         # US Open (32-team draw)
    'First Round':    'Round of 32',
    'Second Round':   'Round of 16',
    'Third Round':    'Quarter-Finals',
    'Quarter-Finals': 'Quarter-Finals',
    'Semi-Finals':    'Semi-Finals',
    'Final':          'Finals',
}

# Tournament metadata for new rows (codes are stable per tournament across years)
# Key: (tournament_label, year_as_str)
# Value: (tournament_code, location, date_string)
TOURNAMENT_META = {
    ("Australian Open", "2024"): (580, "Melbourne,Australia",     "14-28 Jan, 2024"),
    ("Roland Garros",   "2024"): (520, "Paris,France",            "26 May - 9 Jun, 2024"),
    ("Wimbledon",       "2024"): (540, "London,Great Britain",    "1-14 Jul, 2024"),
    ("US Open",         "2024"): (560, "New York,United States",  "26 Aug - 8 Sep, 2024"),
    ("Australian Open", "2025"): (580, "Melbourne,Australia",     "12-26 Jan, 2025"),
    ("Roland Garros",   "2025"): (520, "Paris,France",            "25 May - 8 Jun, 2025"),
    ("Wimbledon",       "2025"): (540, "London,Great Britain",    "30 Jun - 13 Jul, 2025"),
    ("US Open",         "2025"): (560, "New York,United States",  "25 Aug - 7 Sep, 2025"),
    ("Olympics",        "2021"): (None, "Tokyo,Japan",            "24 Jul - 1 Aug, 2021"),
    ("Olympics",        "2024"): (None, "Paris,France",           "27 Jul - 4 Aug, 2024"),
}

print("Config loaded.")

In [ ]:
# ============================================================
# CELL 2: Load both files
# ============================================================
old = pd.read_excel(OLD_FILE)
new = pd.read_excel(NEW_FILE)

print(f"Old : {old.shape}")
print(f"New : {new.shape}")
print(f"Old columns: {list(old.columns)}")

In [ ]:
# ============================================================
# CELL 3: Clean old data
# ============================================================
# Strip trailing spaces from stage names (all old stages end with ' ')
old['stage'] = old['stage'].str.strip()

print("Old stages after strip:", sorted(old['stage'].unique()))
print("Old tournaments:", sorted(old['tournament'].unique()))

In [ ]:
# ============================================================
# CELL 4: Clean new data
# ============================================================

# 4a. Drop Olympics rows
# Only 5 rows per edition with broken team names: the Olympics Wikipedia page
# uses a different bracket template (one entry per stage rather than per match).
n_before = len(new)
new = new[new['Tournament'] != 'Olympics'].copy()
print(f"Dropped {n_before - len(new)} Olympics rows. Remaining: {len(new)}")

# 4b. Fix HTML line-break separators in team names
# Wikipedia encodes doubles pairs as 'Player1<br/>Player2' or 'Player1<br /> Player2'.
def fix_br(s):
    if pd.isna(s):
        return s
    return re.sub(r'<br\s*/?>', ' / ', str(s), flags=re.IGNORECASE).strip()

for col in ['Team1', 'Team2', 'Winner']:
    new[col] = new[col].apply(fix_br)

# 4c. Map Wikipedia stage names to ATP-style Round names
uso_mask = new['Tournament'] == 'US Open'
new['stage'] = new['Stage'].map(GS_STAGE_MAP)                     # default: GS mapping
new.loc[uso_mask, 'stage'] = new.loc[uso_mask, 'Stage'].map(USO_STAGE_MAP)
new['stage'] = new['stage'].fillna(new['Stage'])                   # fallback: keep original

print("\nStage mapping preview:")
print(
    new[['Tournament', 'Stage', 'stage']]
    .drop_duplicates()
    .sort_values(['Tournament', 'stage'])
    .to_string(index=False)
)

In [ ]:
# ============================================================
# CELL 5: Score-parsing helpers
# ============================================================

def parse_score_str(s):
    """
    '7-6(3)' or '7-6' → (7, 6).  Returns (None, None) if unparseable.
    Note: the tiebreak value in parentheses is already stored separately
    in TB_SetN and is not re-extracted here.
    """
    if pd.isna(s) or not str(s).strip():
        return None, None
    m = re.match(r'(\d+)-(\d+)', str(s))
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))


def reshape_row(row):
    """
    Convert one new-format row to the old ATP column schema.

    New format: score from Team1's perspective.
      Score_SetN = 'g1-g2'  (Team1 got g1, Team2 got g2)
      TB_SetN    = set loser's tiebreak points (standard tennis notation)

    Old schema:
      winners_setN / losers_setN  = game counts from winner / loser perspective
      winners_setN_tiebreak       = tiebreak pts when the match winner LOST that
                                    set's tiebreak (i.e. they were the set loser)
      losers_setN_tiebreak        = tiebreak pts when the match loser  LOST that
                                    set's tiebreak (i.e. they were the set loser)

    tournament_code, location, date are filled from TOURNAMENT_META.
    """
    t1_wins = str(row.get('Winner', '')).strip() == str(row.get('Team1', '')).strip()

    def split_pair(s):
        if pd.isna(s) or not str(s).strip():
            return None, None
        parts = [p.strip() for p in re.split(r'\s*/\s*', str(s))]
        return (parts[0] or None), (parts[1] if len(parts) > 1 else None)

    t1p1, t1p2 = split_pair(row['Team1'])
    t2p1, t2p2 = split_pair(row['Team2'])

    wp1, wp2 = (t1p1, t1p2) if t1_wins else (t2p1, t2p2)
    lp1, lp2 = (t2p1, t2p2) if t1_wins else (t1p1, t1p2)

    # Look up tournament metadata
    meta_key = (str(row['Tournament']), str(row['Year']))
    t_code, location, date_str = TOURNAMENT_META.get(meta_key, (None, None, None))

    out = {
        'tournament':      row['Tournament'],
        'location':        location,
        'date':            date_str,
        'year':            row['Year'],
        'tournament_code': t_code,
        'stage':           row['stage'],
        'match_duration':  None,
        'winners_p1':      wp1,
        'winners_p2':      wp2,
        'losers_p1':       lp1,
        'losers_p2':       lp2,
    }

    for n in range(1, 6):
        sc_raw = row.get(f'Score_Set{n}')
        tb_raw = row.get(f'TB_Set{n}')

        g1, g2 = parse_score_str(sc_raw)

        if g1 is None:
            out[f'winners_set{n}'] = None
            out[f'losers_set{n}']  = None
            out[f'winners_set{n}_tiebreak'] = None
            out[f'losers_set{n}_tiebreak']  = None
            continue

        wg = g1 if t1_wins else g2
        lg = g2 if t1_wins else g1
        out[f'winners_set{n}'] = wg
        out[f'losers_set{n}']  = lg

        tb = None if (pd.isna(tb_raw) if tb_raw is not None else True) else int(tb_raw)

        if tb is not None:
            if wg > lg:   # match winner won this set → set loser = match loser
                out[f'winners_set{n}_tiebreak'] = None
                out[f'losers_set{n}_tiebreak']  = tb
            else:         # match winner lost this set → set loser = match winner
                out[f'winners_set{n}_tiebreak'] = tb
                out[f'losers_set{n}_tiebreak']  = None
        else:
            out[f'winners_set{n}_tiebreak'] = None
            out[f'losers_set{n}_tiebreak']  = None

    return out

print("Helpers defined.")

In [ ]:
# ============================================================
# CELL 6: Reshape new data rows
# ============================================================
records = [reshape_row(row) for _, row in new.iterrows()]
new_shaped = pd.DataFrame(records)

print(f"Reshaped: {new_shaped.shape}")
print(new_shaped[['tournament','year','stage','winners_p1','winners_p2',
                   'losers_p1','losers_p2','winners_set1','losers_set1',
                   'winners_set1_tiebreak','losers_set1_tiebreak']].head(5).to_string())

In [ ]:
# ============================================================
# CELL 7: Align columns and merge
# ============================================================

# The old schema has a specific column order (30 columns); we match it exactly.
# Note: old data has no 'winners_set5_tiebreak' column — reproduce this asymmetry.
OLD_COLS = list(old.columns)

for col in OLD_COLS:
    if col not in new_shaped.columns:
        new_shaped[col] = None

new_shaped = new_shaped[OLD_COLS]

merged = pd.concat([old, new_shaped], ignore_index=True)

# Sort: year asc, then tournament, then stage (stable keeps match order within stage)
STAGE_ORDER = {
    'Round of 64': 1, 'Round of 32': 2, 'Round of 16': 3,
    'Quarter-Finals': 4, 'Semi-Finals': 5, 'Finals': 6,
    '1st Round Qualifying': 0, '2nd Round Qualifying': 0,
}
merged['_stage_ord'] = merged['stage'].map(STAGE_ORDER).fillna(3)
merged = merged.sort_values(['year', 'tournament', '_stage_ord'], kind='stable')
merged = merged.drop(columns=['_stage_ord']).reset_index(drop=True)

print(f"Merged shape: {merged.shape}")
print()
print(merged.groupby(['year', 'tournament']).size().rename('matches').to_string())

In [ ]:
# ============================================================
# CELL 8: Validation
# ============================================================
print("=== Stage values in merged file ===")
print(sorted(merged['stage'].dropna().unique()))

print("\n=== Missing player names (new rows only) ===")
new_rows = merged[merged['year'] >= 2024]
print(f"  winners_p1 missing: {new_rows['winners_p1'].isna().sum()} / {len(new_rows)}")
print(f"  losers_p1  missing: {new_rows['losers_p1'].isna().sum()} / {len(new_rows)}")

print("\n=== Tiebreak coverage (new rows) ===")
tb_cols = [c for c in merged.columns if 'tiebreak' in c]
has_tb = new_rows[tb_cols].notna().any(axis=1)
print(f"  Matches with at least one tiebreak: {has_tb.sum()} / {len(new_rows)}")

print("\n=== Sample tiebreak rows (new data) ===")
sample = new_rows[has_tb][['tournament','year','stage','winners_p1','losers_p1',
                            'winners_set1','losers_set1','losers_set1_tiebreak',
                            'winners_set2','losers_set2','winners_set2_tiebreak',
                            'winners_set3','losers_set3','losers_set3_tiebreak']].head(8)
print(sample.to_string())

In [ ]:
# ============================================================
# CELL 9: Save
# ============================================================
merged.to_excel(OUT_FILE, index=False)
print(f"Saved {len(merged)} rows → {OUT_FILE}")
print(f"  Old rows : {len(old)}")
print(f"  New rows : {len(new_shaped)}")